In [2]:
import copy
import os.path
import torch
import torch.functional as F
import math
import logging
import numpy as np
import PIL.Image    
import exiftool
import cv2
import ultralytics
from shapely.geometry import Polygon
from flat_bug.ml_utils import iou_match_pairs, iou
from flat_bug.geometry_simples import contours_to_masks
from ultralytics import YOLO
from ultralytics.nn.tasks import SegmentationModel
from ultralytics.nn.autobackend import AutoBackend
from ultralytics.models.yolo.segment import SegmentationPredictor
from ultralytics.engine.results import Results
from ultralytics.utils import ops

from torchvision.io import read_image
import torchvision.transforms as transforms

from tqdm import tqdm

import logging
import os
from flat_bug.predictor import Predictor
from pyremotedata.implicit_mount import *
from pyremotedata.dataloader import *

import matplotlib.pyplot as plt
import matplotlib as mpl

import itertools

from src.flat_bug.predictor import *
from src.flat_bug.yolo_helpers import *

In [130]:
# 20220820030000-113-snapshot.jpg
# 20230630001000-59-snapshot.jpg
# original_20220730023959-130-snapshot.jpg
# path = "dev/input/20220820030000-113-snapshot.jpg"
# path = "dev/input/20230630001000-59-snapshot.jpg"
path = "dev/input/original_20220730023959-130-snapshot.jpg"
weights = "best.pt"
device = torch.device("cuda:0")
dtype = torch.float16

image = read_image(path).to(device, dtype)
# image = resize(image)
resize = transforms.Resize((1024, 1024))

_model = Predictor(weights, device=device, dtype=dtype)
_model.MINIMUM_TILE_OVERLAP = 384
_model.SCORE_THRESHOLD = 0.2
_model.TIME = False

Transferred 537/537 items from pretrained weights
YOLOv8m-seg summary (fused): 245 layers, 27222963 parameters, 0 gradients, 110.0 GFLOPs


In [131]:
n = 10
start = time.time()
for _ in range(n):
    test = _model.pyramid_predictions(image, path, scale_increment=3/4, scale_before=1/2)
print(f'Average time: {(time.time() - start) / n:.3f}s')

Average time: 0.205s


In [181]:
def test_plot_opencv(obj, linewidth=2, masks=True, boxes=True, conf=True, outpath=None, scale=1):
    # Convert torch tensor to numpy array
    image = obj.image.round().to(torch.uint8).permute(1, 2, 0).cpu().numpy()
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if scale != 1:
        image = cv2.resize(image, (0, 0), fx=scale, fy=scale)

    # Draw masks
    if masks:
        overlay = image.copy()
        # Fill the contours with a semi-transparent red overlay
        contours = [None] * len(obj.masks.data)
        for i, mask in enumerate(obj.masks.data):
            contour = cv2.findContours(mask.to(torch.uint8).cpu().numpy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
            # Calculate areas of each contour
            areas = np.array([cv2.contourArea(c) for c in contour])
            # Select the largest contour and convert it to a tensor
            contour = torch.tensor(contour[np.argmax(areas)], device=obj.device).long().squeeze(1)
            # Convert contour to image coordinates
            contour = obj.contour_to_image_coordinates(contour * scale, interpolate=False).cpu().numpy()
            # Append contour to list of contours
            contours[i] = contour
            cv2.fillPoly(overlay, pts=[contour], color=(0, 0, 255))
        cv2.addWeighted(overlay, 0.3, image, 0.7, 0, image)
        # Draw the contours
        cv2.polylines(image, pts=contours, isClosed=True, color=(0, 0, 255), thickness=linewidth)


    # Draw boxes and confidences
    if boxes:
        for box, conf in zip(obj.boxes, obj.confs):
            box = (box * scale)
            box[:2] = box[:2].floor()
            box[2:] = box[2:].ceil()
            box = box.long()
            start_point = (int(box[0]), int(box[1]))
            end_point = (int(box[2]), int(box[3]))
            cv2.rectangle(image, start_point, end_point, (0, 0, 0), linewidth)  # Red box
            if conf:
                cv2.putText(image, f"{conf*100:.3g}%", (start_point[0], start_point[1] - round(10 * scale)),
                            cv2.FONT_HERSHEY_SIMPLEX, 1 * scale, (0, 0, 0), max(1, round(2 * scale)))

    # Save or show the image
    if outpath:
        cv2.imwrite(outpath, image)
    else:
        cv2.imshow('Image', image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

In [185]:
"drwxr-xr-x sweden/0ba6f45e (SE1)/".split(" ", 1)

['drwxr-xr-x', 'sweden/0ba6f45e (SE1)/']

In [182]:
test_plot_opencv(test, outpath="test/test.jpg", scale=1/2, linewidth=2)

In [170]:
with_torch = []
fast = []
full = []
n = 10
for _ in range(n):
    start = time.time()
    test.plot_torch(scale=1/2, outpath="test/test.jpg")
    with_torch.append(time.time() - start)
    start = time.time()
    test_plot_opencv(test, outpath="test/test.jpg", scale=1/2)
    fast.append(time.time() - start)
    start = time.time()
    test_plot_opencv(test, outpath="test/test.jpg")
    full.append(time.time() - start)
print(f'Average with torch time: {sum(with_torch)/n:.3f}s')
print(f'Average fast time: {sum(fast)/n:.3f}s')
print(f'Average full time: {sum(full)/n:.3f}s')

Average with torch time: 0.120s
Average fast time: 0.078s
Average full time: 0.204s


In [6]:
for mask in test.masks.data:
    contour = cv2.findContours(mask.to(torch.uint8).cpu().numpy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0]
    contour = np.concatenate(contour, axis=0).reshape(-1, 2)
    print(contour)
    break

[[119 112]
 [119 113]
 [119 114]
 [120 115]
 [120 116]
 [120 117]
 [119 118]
 [119 119]
 [119 120]
 [119 121]
 [120 122]
 [121 122]
 [122 122]
 [123 122]
 [124 122]
 [125 122]
 [126 122]
 [126 121]
 [126 120]
 [126 119]
 [126 118]
 [125 118]
 [124 118]
 [123 118]
 [122 117]
 [122 116]
 [122 115]
 [121 114]
 [121 113]
 [121 112]
 [120 112]
 [125  90]
 [125  91]
 [124  92]
 [124  93]
 [124  94]
 [124  95]
 [123  96]
 [123  97]
 [122  98]
 [121  98]
 [120  99]
 [119 100]
 [119 101]
 [119 102]
 [120 103]
 [121 104]
 [122 105]
 [123 106]
 [123 107]
 [123 108]
 [124 108]
 [125 108]
 [126 109]
 [127 110]
 [128 111]
 [128 112]
 [127 113]
 [127 114]
 [128 113]
 [129 114]
 [130 115]
 [129 116]
 [130 116]
 [131 117]
 [132 117]
 [133 117]
 [134 117]
 [135 118]
 [136 118]
 [137 119]
 [138 119]
 [139 119]
 [140 119]
 [141 118]
 [141 117]
 [141 116]
 [141 115]
 [142 114]
 [142 113]
 [142 112]
 [142 111]
 [141 110]
 [141 109]
 [142 108]
 [143 107]
 [144 106]
 [145 106]
 [145 105]
 [144 104]
 [143 104]

In [33]:
test_save_path = test.save("test", fast=True)
test_restored = TensorPredictions().load("test/20220820030000-113-snapshot/20220820030000-113-snapshot.json")
test_restored.image = test.image
test_restored.plot_matplotlib()

In [ ]:
# from flat_bug.geometry_simples import remove_inserted_points

# tcont = (torch.tensor([1000, 1000]).cuda() + torch.stack(torch.meshgrid(torch.arange(50), torch.arange(50))).reshape(2, -1).T.cuda()).long()
# # tcont = remove_inserted_points(tcont)

# start = time.time()
# tnew_order = test_order_points_clockwise(tcont.clone())
# print(f'New order took {time.time() - start} seconds')
# start = time.time()
# told_order = order_points_clockwise(tcont.clone())
# print(f'Old order took {time.time() - start} seconds')

# tpold_order = told_order[0].cpu()
# tpnew_order = tnew_order[0].cpu()

# fig, axs = plt.subplots(1, 2)
# axs[0].plot(tpold_order[:, 0], tpold_order[:, 1])
# axs[1].plot(tpnew_order[:, 0], tpnew_order[:, 1])
# plt.show()
# told_order, tnew_order

fig, axs = plt.subplots(1, 3, figsize=(10, 5))
mask_idx = 3
mask = test.masks.data[mask_idx]
contour = find_contours(test.masks.data[mask_idx], largest_only=True)
bbox = torch.cat([contour.min(dim=0)[0], contour.max(dim=0)[0]]).long()
idx = torch.arange(len(contour))

axs[0].imshow(contours_to_masks([contour], *mask.shape).cpu()[0, bbox[0]:bbox[2], bbox[1]:bbox[3]].T)
axs[1].imshow(mask.cpu()[bbox[0]:bbox[2], bbox[1]:bbox[3]].T)
axs[2].scatter(contour[:, 0].cpu(), contour[:, 1].cpu(), c=idx, cmap='viridis')
axs[2].invert_yaxis()
# axs[2].invert_xaxis()
for ax in axs:
    ax.axis('off')
    ax.set_aspect('equal')

In [26]:
tkern = torch.ones((3, 3), dtype=torch.float32).unsqueeze(0).unsqueeze(0)
tmask = torch.zeros((100, 100), dtype=torch.bool)
tmask[10:20, 10:20] = True
tmask[30:40, 30:40] = True
tmask[5:25, 15] = True
tmask[15, 5:25] = True

tbound = (F.conv2d(tmask.unsqueeze(0).unsqueeze(0).float(), tkern, padding=1).squeeze() < 9) & tmask

fig, axs = plt.subplots(1, 2)

axs[0].imshow(tbound)
axs[1].imshow(tmask)

In [ ]:
# def is_in_2d(point: torch.Tensor, points: torch.Tensor) -> bool:
#     """
#     Checks if a point is in a set of points. Does this by checking the x-coordinates and y

#     Args:
#         point (torch.Tensor): The point of size (,2) to check.
#         points (torch.Tensor): The set of points of size (N, 2) to check against.

#     Returns:
#         bool: True if the point is in the set of points, False otherwise.
#     """
#     return (point == points).all(dim=1).any()

# def order_points_clockwise(boundary_indices: torch.Tensor) -> torch.Tensor:
#     """
#     Orders the boundary points in a clockwise manner.

#     Args:
#         boundary_indices (torch.Tensor): Coordinates of the sparse boundary pixels of size (N, 2).

#     Returns:
#         torch.Tensor: Coordinates of the boundary pixels ordered clockwise of size (N, 2).
#     """
#     if len(boundary_indices) <= 1:
#         return [boundary_indices]
#     device = boundary_indices.device
#     # Find the top-leftmost point as the starting point
#     start_point = boundary_indices[boundary_indices[:, 0] == boundary_indices[:, 0].min()]
#     start_point = start_point[start_point[:, 1].argmax()]

#     # Directions to move: right, down, left, up (clockwise)
#     directions = torch.tensor([[0, 1], [1, 0], [0, -1], [-1, 0]], dtype=torch.long, device=device)

#     # Initialize helpers
#     ordered_points = torch.ones_like(boundary_indices) * -1
#     ordered_points[0] = start_point
#     current_point = start_point
#     dir_idx = 1
#     i = 1
#     other_regions = []

#     while i < len(boundary_indices):
#         found_next = False
#         for d in range(4):
#             this_dir = (dir_idx + d) % 4 # Always start with the current direction and go clockwise
#             next_point = current_point + directions[this_dir]
#             if is_in_2d(next_point, boundary_indices) and not is_in_2d(next_point, ordered_points[:i]):
#                 ordered_points[i] = next_point
#                 current_point = next_point
#                 found_next = True
#                 dir_idx = this_dir - 1 # Update to the new direction
#                 i += 1
#                 break
#         if not found_next and i != len(boundary_indices):
#             # This happens when there are more than one contiguous regions in the mask
#             # Here we simply call recursively on the remaining points
#             remaining_points = boundary_indices[~torch.stack([is_in_2d(p, ordered_points[:i]) for p in boundary_indices])]
#             other_regions = order_points_clockwise(remaining_points)
#             break
#         if i == len(boundary_indices):
#             break

#     return [ordered_points[:i]] + other_regions

# def duplicate_rows_and_columns(mask: torch.Tensor) -> torch.Tensor:
#     """
#     Expands a mask by a factor of two, by duplicating rows and columns.  

#     Args:
#         mask (torch.Tensor): The mask of size (H, W) to expand.

#     Returns:
#         torch.Tensor: The expanded mask of size (2H, 2W).
#     """
#     # Duplicate rows
#     expanded_rows = torch.repeat_interleave(mask, 2, dim=0)
#     # Duplicate columns
#     expanded_mask = torch.repeat_interleave(expanded_rows, 2, dim=1)
#     # Return the expanded mask
#     return expanded_mask

# def find_first_occurrences_1d(elements: torch.Tensor) -> torch.Tensor:
#     """
#     Finds the indices of the first occurrences of each unique element in a 1D tensor.

#     Args:
#         elements (torch.Tensor): The elements to find the first occurrences of.
    
#     Returns:
#         torch.Tensor: The indices of the first occurrences of each unique element.
#     """
#     unique_elements = elements.unique()
#     return torch.stack([torch.where(elements == e)[0][0] for e in unique_elements])

# def first_indices_of_unique_elements(elements_multidimensional: torch.Tensor, dim: int=0) -> torch.Tensor:
#     """
#     Finds the first indices of unique elements in a multidimensional tensor along a given dimension.
#     """
#     # Find the indices of each group of unique elements along the given dimension
#     _, unique_idx = torch.unique(elements_multidimensional, return_inverse=True, dim=dim)
#     # Find the index of the first occurrence of each unique element group
#     return find_first_occurrences_1d(unique_idx)

# def remove_inserted_points(boundary_indices: torch.Tensor) -> torch.Tensor:
#     """
#     Removes the inserted points from the boundary indices. Indices where either coordinates are even are considered valid, the rest are removed. The valid indices are then divided by two to get the original indices.
#     """
#     # Divide by 2 to map back to original coordinates
#     scaled_down_points = boundary_indices // 2
#     # Remove duplicates that may have been created by scaling down
#     unique_idx = first_indices_of_unique_elements(scaled_down_points)
#     # Return the unique points
#     return scaled_down_points[unique_idx.sort().values]


# def find_contiguous_regions(mask: torch.Tensor) -> torch.Tensor:
#     """
#     Efficiently finds the sparse boundary of a contiguous region in the mask.
    
#     Args:
#         mask (torch.Tensor): A 2D boolean tensor representing the mask.

#     Returns:
#         torch.Tensor: Coordinates of the sparse boundary pixels.
#     """
#     device = mask.device

#     # Duplicate rows and columns to make sure that the all boundaries have two valid neighbors
#     mask = duplicate_rows_and_columns(mask)

#     # Kernel to check for 8-neighbors
#     kernel = torch.ones((3, 3), dtype=torch.float, device=device).unsqueeze(0).unsqueeze(0)

#     # Convolve with the kernel to count neighbors
#     neighbor_count = F.conv2d(mask.float().unsqueeze(0).unsqueeze(0), kernel, padding=1).squeeze()

#     # Boundary pixels are those in the original mask with less than 9 neighbors
#     boundary = (neighbor_count < 9) & mask

#     # Extract coordinates of boundary pixels
#     boundary_indices = torch.nonzero(boundary, as_tuple=False)

#     # Find the regions
#     regions = order_points_clockwise(boundary_indices)

#     # Remove the inserted points
#     regions = [remove_inserted_points(r) for r in regions]

#     return regions


In [ ]:
# def remove_unconnected_points(contour : torch.Tensor) -> torch.Tensor:
#     """
#     Takes a contour represented as (i, j) index-coordinates in a Nx2 tensor and removes unconnected points. A point is connected if the next element in the contour is a neighbor of the current element in the cardinal directions.

#     Args:
#         contour (torch.Tensor): Contour represented as (i, j) index-coordinates in a Nx2 tensor

#     Returns:
#         torch.Tensor: Contour with unconnected points removed of size Mx2, where M <= N
#     """
#     # If there are one or fewer points, return the contour
#     if contour.shape[0] <= 1:
#         return contour
#     # Clone the contour to avoid modifying the original
#     contour = contour.clone()
#     # Remove the unconnected points, which may expose new unconnected points
#     while True:
#         # Compute the manhattan distance to the next point
#         distance_to_next = (contour - contour.roll(-1, 0)).abs()
#         # The manhattan distance is 1 if and only if the points are neighbors in the cardinal directions
#         connects = (distance_to_next.sum(dim=1) == 1)
#         # If all points are connected, we are done
#         if connects.all():
#             break
#         # Otherwise, remove the unconnected points
#         contour = contour[connects]
#         # If there are one or fewer points, break
#         if contour.shape[0] <= 1:
#             break
#     return contour

# def test_contours_to_masks(contours : list[torch.Tensor], height : int, width : int, dtype : torch.dtype) -> torch.Tensor:
#     """
#     Takes a list of contours represented as (i, j) index-coordinates in a Xx2 tensor and returns a NxHxW tensor of boolean masks with the contours filled in.

#     Args:
#         contours (list[torch.Tensor]): List of contours represented as (i, j) index-coordinates in a Nx2 tensor (OBS: dtype=torch.long)
#         height (int): Height of the masks
#         width (int): Width of the masks

#     Returns:
#         torch.Tensor: NxHxW tensor of boolean masks with the contours filled in
#     """
#     device = contours[0].device
#     N = len(contours)
#     # Type checking
#     assert all(c.dtype == torch.long for c in contours), "All contours must be of dtype=torch.long"
#     assert all(c.device == device for c in contours), "All contours must be on the same device"
#     assert all(len(c.shape) == 2 and c.shape[1] == 2 for c in contours), "All contours must be Xx2 tensors"
#     assert isinstance(height, int) and isinstance(width, int), "Height and width must be integers"
#     assert height > 0 and width > 0, "Height and width must be positive"

#     # Initialize the masks
#     masks = torch.zeros((N, height, width), dtype=torch.bool, device=device)
#     if N == 0:
#         return masks

#     # Filling in the masks
#     for i, contour in enumerate(contours):
#         # Remove unconnected points
#         trimmed_contour = remove_unconnected_points(contour)
#         # Create a mask with the trimmed contour filled in
#         contour_mask = torch.zeros((height, width), dtype=torch.bool, device=device)
#         contour_mask[trimmed_contour[:, 0], trimmed_contour[:, 1]] = True
#         # The smallest polygon with a hole has 9 points
#         if len(trimmed_contour) > 9:
#             # For each row, fill in the pixels between each consecutive pair of contour pixels
#             for j, row in enumerate(contour_mask):
#                 # Find the indices of the contour pixels in this row
#                 contour_pixels = torch.nonzero(row, as_tuple=False)
#                 # Remove consecutive neighbors (following a horizontal line)
#                 contour_pixels = contour_pixels[torch.abs(contour_pixels.roll(-1) - contour_pixels) > 1]
#                 # Fill in the pixels between each pair of contour pixels
#                 for k in range(0, len(contour_pixels) - 1, 2):
#                     xmin, xmax = contour_pixels[k], contour_pixels[k + 1] + 1
#                     contour_mask[j, xmin:xmax] = True
#         # Remember to add the points which were removed by remove_unconnected_points
#         contour_mask[contour[:, 0], contour[:, 1]] = True
#         # Add the mask to the list of masks
#         masks[i] = contour_mask

#     return masks


In [ ]:
tci, tch, tcw = test.image.shape # Original image shape
tc, th, tw = test.masks.data.shape # Mask shape
cont = test.contours[0] # Contour in the format of a ndarray of shape (n, 2) where n is the number of points and 2 is the x and y coordinates
cont[:, 0] /= tch # Scale the x coordinates to the original image
cont[:, 1] /= tcw # Scale the y coordinates to the original image
cont[:, 0] *= tw # Scale the x coordinates to the mask
cont[:, 1] *= th # Scale the y coordinates to the mask
# cont = cont[::-1]
cont = torch.tensor(cont.copy(), dtype=torch.long)
cont = expand_contour(cont)

# Convert the contour to a mask
mask = test_contours_to_masks([cont], th, tw, dtype=torch.float32)[0]

cont_idx = np.arange(len(cont)) # Create an index for the contour

cont = cont.cpu().numpy() # Convert the contour to a numpy array
cont = cont[:, ::-1] # Flip the coordinates to be in the format of (x, y)

# Plot the contour with points colored by the index
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(mask.cpu().numpy(), cmap="gray")
ax.scatter(cont[:, 0], cont[:, 1], c=cont_idx, cmap="viridis")
ax.set_aspect("equal")
ax.set_title("Contour with color")
plt.show()


tensor(0) tensor(1) tensor([222.]) tensor([50.])
tensor(-2) tensor(0) tensor([222., 220.]) tensor([51., 51.])
tensor(0) tensor(2) tensor([220., 220.]) tensor([51., 53.])
tensor(-3) tensor(1) tensor([220.0000, 218.5000, 217.0000]) tensor([53.0000, 53.5000, 54.0000])
tensor(0) tensor(0) tensor([]) tensor([])
tensor(-3) tensor(1) tensor([218.0000, 216.5000, 215.0000]) tensor([54.0000, 54.5000, 55.0000])
tensor(-1) tensor(0) tensor([216.]) tensor([55.])
tensor(-5) tensor(1) tensor([215.0000, 213.7500, 212.5000, 211.2500, 210.0000]) tensor([55.0000, 55.2500, 55.5000, 55.7500, 56.0000])
tensor(0) tensor(1) tensor([211.]) tensor([56.])
tensor(7) tensor(2) tensor([211.0000, 212.1667, 213.3333, 214.5000, 215.6667, 216.8333, 218.0000]) tensor([57.0000, 57.3333, 57.6667, 58.0000, 58.3333, 58.6667, 59.0000])
tensor(0) tensor(1) tensor([218.]) tensor([59.])
tensor(4) tensor(0) tensor([218.0000, 219.3333, 220.6667, 222.0000]) tensor([60., 60., 60., 60.])
tensor(5) tensor(2) tensor([222.0000, 223.250

AssertionError: All contours must be of dtype=torch.long

In [ ]:
mask[50:70, 210:230]

tensor([[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False, False, False,  True, False,  True, False,  True, False,  True, False, False],
        [False, False, False, False, False, False, False, False, False,  True, False,  True, False,  True, False,  True, False,  True, False, False],
        [False, False, False, False, False, False, False, False,  True, False,  True, False,  True, False,  True, False,  True, False,  True,  True],
        [False, False, False, False, False, False, False, False,  True, False,  True, False,  True, False,  True, False,  True, False,  True, False],
        [False, False, False, False, False, False,  True, False,  True, False,  True, False,  True, False,  True, False,  True, False,  True, False],
        [False,  True, False,  True, False,  True, False,  True, False,  True, False,  True, False, 

In [ ]:
tci, tch, tcw = test.image.shape # Original image shape
tc, th, tw = test.masks.data.shape # Mask shape
cont = test.contours[0] # Contour in the format of a ndarray of shape (n, 2) where n is the number of points and 2 is the x and y coordinates
cont[:, 0] /= tch # Scale the x coordinates to the original image
cont[:, 1] /= tcw # Scale the y coordinates to the original image
cont[:, 0] *= tw # Scale the x coordinates to the mask
cont[:, 1] *= th # Scale the y coordinates to the mask
cont = cont[::-1]
cont = torch.tensor(cont.copy(), dtype=torch.long)
# cont = expand_contour(cont)

# Convert the contour to a mask
mask = test_contours_to_masks([cont], th, tw, dtype=torch.float32)[0]

cont_idx = np.arange(len(cont)) # Create an index for the contour

cont = cont.cpu().numpy() # Convert the contour to a numpy array
cont = cont[:, ::-1] # Flip the coordinates to be in the format of (x, y)

# Plot the contour with points colored by the index
fig, ax = plt.subplots(figsize=(10, 10))
# ax.imshow(mask.cpu().numpy(), cmap="gray")
ax.scatter(cont[:, 0], cont[:, 1], c=cont_idx, cmap="viridis")
ax.set_aspect("equal")
ax.set_title("Contour with color")
plt.show()


0 2 tensor([229, 229]) tensor([50, 52])
-2 0 tensor([229, 227]) tensor([52, 52])
0 1 tensor([227]) tensor([52])
5 1 tensor([227, 228, 229, 230, 232]) tensor([53, 53, 53, 53, 54])
9 0 tensor([231, 232, 233, 234, 235, 236, 237, 238, 240]) tensor([54, 54, 54, 54, 54, 54, 54, 54, 54])
1 0 tensor([240]) tensor([54])
4 0 tensor([241, 242, 243, 245]) tensor([54, 54, 54, 54])
2 0 tensor([245, 247]) tensor([54, 54])
0 1 tensor([247]) tensor([54])
1 0 tensor([247]) tensor([55])
0 1 tensor([248]) tensor([55])
3 1 tensor([248, 249, 251]) tensor([56, 56, 57])
0 0 tensor([], dtype=torch.int64) tensor([], dtype=torch.int64)
3 1 tensor([250, 251, 253]) tensor([57, 57, 58])
4 0 tensor([252, 253, 254, 256]) tensor([58, 58, 58, 58])
0 1 tensor([256]) tensor([58])
0 0 tensor([], dtype=torch.int64) tensor([], dtype=torch.int64)
-1 0 tensor([257]) tensor([59])
-6 2 tensor([256, 254, 253, 252, 251, 250]) tensor([59, 59, 59, 60, 60, 61])
0 0 tensor([], dtype=torch.int64) tensor([], dtype=torch.int64)
3 1 tens

In [ ]:
torch.load("test/serialized.pt")

In [ ]:
def find_contours(neighbors):
    _device = neighbors[0].device
    outer_idx = torch.tensor([i for i, n in enumerate(neighbors) if len(n) < 9], dtype=torch.long, device=_device)
    print(len(neighbors), len(outer_idx))
    inner_idx = torch.tensor([i for i in range(len(neighbors)) if i not in outer_idx], dtype=torch.long, device=_device)
    outer_points = [neighbors[i][torch.isin(neighbors[i], outer_idx)] for i in outer_idx]
    # Remap indices for outer points
    outer_remap = torch.arange(len(neighbors), dtype=torch.long, device=_device)
    outer_remap[outer_idx] = torch.arange(len(outer_idx), dtype=torch.long, device=_device)
    outer_remap[inner_idx] = -1
    outer_points = [outer_remap[o] for o in outer_points]
    
    skippers = torch.zeros(len(neighbors), dtype=torch.bool, device=_device)
    winners = skippers.clone()

    group_ind = 0
    while group_ind < len(outer_points):
        if skippers[group_ind]:
            group_ind += 1
            continue
        last_added = outer_points[group_ind]
        skippers[group_ind] = True
        winners[group_ind] = True
        while True:
            this = outer_points[group_ind]
            
            mergers = torch.zeros(len(outer_points), dtype=torch.bool, device=_device)
            new_neighbors = mergers.clone()
            for i, o in enumerate(outer_points):
                if skippers[i]:
                    continue
                old_neighbors = torch.isin(o, last_added, assume_unique=True)
                if not old_neighbors.any():
                    continue
                mergers[i] = True
                skippers[i] = True
                if old_neighbors.all():
                    continue
                new_neighbors[o[~old_neighbors]] = True

            if not mergers.any():
                break
            
            last_added = torch.where(new_neighbors)[0].unique()
            outer_points[group_ind] = torch.cat([last_added, this])
        group_ind += 1
    return [outer_idx[n] for n, w in zip(outer_points, winners) if w], inner_idx

def find_neighbors(mask, pos):
    raise NotImplementedError("Seems to be some bug with this function, but I cannot reproduce it at the moment. Only happens on real data, as far as I have been able to find.")
    nmask = torch.zeros((mask.shape[0]+2, mask.shape[1]+2), dtype=torch.long, device=mask.device)
    nmask[*(pos + 1).T] = torch.arange(len(pos), device=mask.device, dtype=torch.long) + 1
    nidx = torch.arange(3, device=mask.device, dtype=torch.long).unsqueeze(0).repeat(len(pos), 3) - 1
    nidx += (pos[:, 0].unsqueeze(1) + 1) + (pos[:, 1].unsqueeze(1) + 1) * nmask.shape[0]
    nidx[:, :3] -= nmask.shape[0]
    nidx[:, -3:] += nmask.shape[0]
    # assert (nmask.flatten()[nidx[:, 4]].sort().values == torch.arange(len(pos), device=mask.device, dtype=torch.long) + 1).all(), f"Centers {nidx[:, 4].sort().values} do not match {nmask.flatten().nonzero(as_tuple=False).flatten().sort().values}"

    return [neighbors[neighbors != 0] - 1 for neighbors in nmask.flatten()[nidx]]

def find_neighbors_naive(mask):
    pos = mask.nonzero()
    return [torch.where(((pos[i].unsqueeze(0) - pos) ** 2).sum(dim=1).sqrt() < 1.5)[0] for i in range(len(pos))]
    

def find_contigs(mask):
    start = time.time()
    pos = mask.nonzero(as_tuple=False)
    # neighbors = find_neighbors(mask, pos)
    neighbors = find_neighbors_naive(mask)
    print(neighbors)
    neighbor_finding_time = time.time() - start
    start = time.time()
    contours, inners = find_contours(neighbors)
    print(contours)
    contour_finding_time = time.time() - start
    start = time.time()
    pos = pos.float()
    contours = [pos[c] for c in contours]
    # Assign inner points to contours
    inner_to_contour_min_dist = [(torch.cdist(pos[inners], c)).min(dim=1).values for c in contours]
    inner_to_contour_min_dist = torch.stack(inner_to_contour_min_dist)
    which_contour = inner_to_contour_min_dist.argmin(dim=0)
    for i, c in enumerate(contours):
        c = torch.cat([c, pos[inners[which_contour == i]]])
        contours[i] = c.long()
    inner_assigment_time = time.time() - start
    start = time.time()
    split_masks = torch.zeros((len(contours), *mask.shape), dtype=torch.bool)
    for i, c in enumerate(contours):
        split_masks[i, c[:,0], c[:,1]] = True
    mask_creation_time = time.time() - start
    total_time = neighbor_finding_time + contour_finding_time + inner_assigment_time + mask_creation_time
    print(f'Found {len(split_masks)} in {total_time:.2f} seconds | Neighbors {neighbor_finding_time:.2f} ({neighbor_finding_time/total_time*100:.3g}%) | Contours {contour_finding_time:.2f} ({contour_finding_time/total_time*100:.3g}%) | Inner Assignment {inner_assigment_time:.3f} ({inner_assigment_time/total_time*100:.3g}%) | Mask Creation {mask_creation_time:.3f} ({mask_creation_time/total_time*100:.3g}%)')
    return split_masks

def expand_mask(mask, n=1, dtype=dtype):
    neighbor_kernel = torch.ones(1, 1, 1+2*n, 1+2*n, device=mask.device, dtype=dtype)
    return torch.nn.functional.conv2d(mask.unsqueeze(0).unsqueeze(0).to(dtype), neighbor_kernel, padding=n).squeeze(0).squeeze(0) > 0.5

test.masks.data = test.masks.data.to(torch.device("cpu"))
torch.cat([find_contigs(m) for m in test.masks.data])

# tmask = torch.zeros((200, 200), dtype=torch.bool, device=torch.device("cuda:0"))
# tmask[40:80, 40:80] = True
# tmask[120:180, 120:180] = True

# find_contigs(tmask)


[tensor([0, 1, 5, 6]), tensor([0, 1, 2, 5, 6, 7]), tensor([1, 2, 3, 6, 7, 8]), tensor([2, 3, 4, 7, 8, 9]), tensor([3, 4, 8, 9]), tensor([ 0,  1,  5,  6, 10, 11, 12]), tensor([ 0,  1,  2,  5,  6,  7, 11, 12, 13]), tensor([ 1,  2,  3,  6,  7,  8, 12, 13, 14]), tensor([ 2,  3,  4,  7,  8,  9, 13, 14, 15]), tensor([ 3,  4,  8,  9, 14, 15]), tensor([ 5, 10, 11, 16, 17]), tensor([ 5,  6, 10, 11, 12, 16, 17, 18]), tensor([ 5,  6,  7, 11, 12, 13, 17, 18, 19]), tensor([ 6,  7,  8, 12, 13, 14, 18, 19, 20]), tensor([ 7,  8,  9, 13, 14, 15, 19, 20, 21]), tensor([ 8,  9, 14, 15, 20, 21]), tensor([10, 11, 16, 17, 22, 23]), tensor([10, 11, 12, 16, 17, 18, 22, 23, 24]), tensor([11, 12, 13, 17, 18, 19, 23, 24, 25]), tensor([12, 13, 14, 18, 19, 20, 24, 25, 26]), tensor([13, 14, 15, 19, 20, 21, 25, 26]), tensor([14, 15, 20, 21, 26]), tensor([16, 17, 22, 23, 27, 28]), tensor([16, 17, 18, 22, 23, 24, 27, 28, 29]), tensor([17, 18, 19, 23, 24, 25, 28, 29, 30]), tensor([18, 19, 20, 24, 25, 26, 29, 30, 31]), t

tensor([[[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False]],

        [[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False]],

        [[False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         ...,
         [False, False, False,  ..., False, False, False],
         [

In [ ]:
from flat_bug.geometry_simples import *

tmask = torch.zeros((10, 10), dtype=torch.bool, device=device)
tmask[2:5, 2:5] = True
tmask[6:8, 6:8] = True

pos = tmask.nonzero(as_tuple=False)

start = time.time()
find_neighbors(tmask, pos)
print(f'Efficient: {time.time() - start}')

start = time.time()
find_neighbors_naive(tmask)
print(f'Naive: {time.time() - start}')

find_neighbors(tmask, pos), find_neighbors_naive(tmask)

NotImplementedError: Seems to be some bug with this function, but I cannot reproduce it at the moment. Only happens on real data, as far as I have been able to find.

In [ ]:
tmask.unsqueeze(0).repeat(2, 1, 1).sum(dim=(1, 2))

tensor([13, 13], device='cuda:0')

In [ ]:
tmask = torch.zeros((10, 10), dtype=torch.bool, device=device)
tmask[2:5, 2:5] = True
tmask[6:8, 6:8] = True

tpos = tmask.nonzero(as_tuple=False)

tmaskl = torch.zeros((12, 12), dtype=torch.long, device=device)
tmaskl[*(tpos + 1).T] = torch.arange(len(tpos), device=device, dtype=torch.long) + 1
# tminx, tmaxx, tminy, tmaxy = tpos[:, 0] - 1, tpos[:, 0] + 2, tpos[:, 1] - 1, tpos[:, 1] + 2

tidx = torch.arange(3, device=device, dtype=torch.long).unsqueeze(0).repeat(len(tpos), 3) - 1
tidx += (tpos[:, 0].unsqueeze(1) + 1) + (tpos[:, 1].unsqueeze(1) + 1) * tmaskl.shape[0]
tidx[:, :3] -= tmaskl.shape[0]
tidx[:, -3:] += tmaskl.shape[0]

print(tmaskl)
tmaskl.flatten()[tidx]

# tidx, tmaskl.flatten().nonzero(), tmaskl.nonzero(as_tuple=False), tpos[:, 0] + 1 + (tpos[:, 1] + 1) * tmaskl.shape[0]

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  1,  2,  3,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  4,  5,  6,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  7,  8,  9,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0, 10, 11,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0, 12, 13,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]])


tensor([[ 0,  0,  0,  0,  1,  2,  0,  4,  5],
        [ 0,  1,  2,  0,  4,  5,  0,  7,  8],
        [ 0,  4,  5,  0,  7,  8,  0,  0,  0],
        [ 0,  0,  0,  1,  2,  3,  4,  5,  6],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9],
        [ 4,  5,  6,  7,  8,  9,  0,  0,  0],
        [ 0,  0,  0,  2,  3,  0,  5,  6,  0],
        [ 2,  3,  0,  5,  6,  0,  8,  9,  0],
        [ 5,  6,  0,  8,  9,  0,  0,  0,  0],
        [ 0,  0,  0,  0, 10, 11,  0, 12, 13],
        [ 0, 10, 11,  0, 12, 13,  0,  0,  0],
        [ 0,  0,  0, 10, 11,  0, 12, 13,  0],
        [10, 11,  0, 12, 13,  0,  0,  0,  0]])

In [ ]:
(tpos[:, 0].unsqueeze(1) + 1)[-3:]

tensor([[7],
        [8],
        [8]])

In [ ]:
tmask[*tmask.nonzero(as_tuple=False).T]

tensor([True, True, True, True, True, True, True, True, True, True, True, True, True], device='cuda:0')

In [ ]:
test.plot_matplotlib()

In [ ]:
# test.plot(dpi=300, scale=1/2, outpath="test/test_plot.png")
test.plot_opencv(outpath="test/test_plot_opencv.png")

In [ ]:
def expand_with_neighbors(indices, mx, my):
    # Initialize a list to store the expanded indices
    expanded_indices = []

    # Define the relative positions of the neighbors
    neighbors = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 0), (0, 1), (1, -1), (1, 0), (1, 1)]

    # Iterate over each index pair and add its neighbors
    for x, y in indices:
        for dx, dy in neighbors:
            new_x, new_y = x + dx, y + dy
            new_x = min(max(new_x, 0), mx)
            new_y = min(max(new_y, 0), my)
            expanded_indices.append([new_x, new_y])

    # Convert the list of indices to a numpy array
    return np.array(expanded_indices)


In [ ]:
plt.figure(figsize=(10, 10))
ms = np.zeros(image.shape[1:]) # CxHxW
boxes = test._bboxes # x, y, w, h
print(ms.shape)
xys = np.concatenate(test.contours)
msx, msy = ms.shape
xys = expand_with_neighbors(expand_with_neighbors(xys, msy - 1, msx - 1), msy - 1, msx - 1)
ms[xys[:, 1], xys[:, 0]] = 1
# Permute to HxWxC (numpy)
# ms = ms.transpose(1, 2, 0)
plt.imshow(image.cpu().int().permute(1, 2, 0))
# plt.imshow(ms, alpha = .75)
for x, y, w, h in boxes:
    plt.gca().add_patch(mpl.patches.Rectangle((x, y), w, h, fill=False, edgecolor='r', linewidth=1))
plt.show()

In [ ]:
def intersect_test(rect1, rect2s, area_only=False, debug=False):
    """
    Calculates the intersection of a rectangle with a set of rectangles.
    """
    if len(rect1.shape) == 1 and not rect1.shape[0] == 4 or len(rect1.shape) == 2 and not rect1.shape[1] == 4:
        raise ValueError(f"Rectangles must be of shape (n, 4), not {rect1.shape}")
    if len(rect2s.shape) == 1 and not rect2s.shape[0] == 4 or len(rect2s.shape) == 2 and not rect2s.shape[1] == 4:
        raise ValueError(f"Rectangles must be of shape (n, 4), not {rect2s.shape}")
    if len(rect1.shape) == 1:
        rect1 = rect1.unsqueeze(0)
    if len(rect2s.shape) == 1:
        rect2s = rect2s.unsqueeze(0)

    # Safer to enable this, but it is slower
    # # Check the validity of the rectangles
    # if not check_bltr_validity(rect1, debug):
    #     rect1 = fix2btlr(rect1)
    # if not check_bltr_validity(rect2s, debug):
    #     rect2s = fix2btlr(rect2s)

    # Calculate vectors from each corner of rect1 to each corner of rect2s
    blbltrtr = rect2s - rect1
    bl_to_bl = blbltrtr[:, :2]
    tr_to_tr = blbltrtr[:, 2:] 
    bltrtrbl = rect2s[:, [2, 3, 0, 1]] - rect1
    bl_to_tr = bltrtrbl[:, :2]
    tr_to_bl = bltrtrbl[:, 2:]
    
    # Determine if each corner of rect1 is inside each rect2
    inside_tr = (tr_to_tr[:, 0] >= 0) & (tr_to_tr[:, 1] >= 0) & (tr_to_bl[:, 0] <= 0) & (tr_to_bl[:, 1] <= 0)
    inside_bl = (bl_to_bl[:, 0] <= 0) & (bl_to_bl[:, 1] <= 0) & (bl_to_tr[:, 0] >= 0) & (bl_to_tr[:, 1] >= 0)
    inside_tl = (bl_to_bl[:, 0] <= 0) & (tr_to_tr[:, 1] >= 0) & (bl_to_tr[:, 0] >= 0) & (tr_to_bl[:, 1] <= 0)
    inside_br = (tr_to_tr[:, 0] >= 0) & (bl_to_bl[:, 1] <= 0) & (tr_to_bl[:, 0] <= 0) & (bl_to_tr[:, 1] >= 0)

    # Check for enclosure
    enclosure = (rect1[:, :2] <= rect2s[:, :2]) & (rect1[:, 2:] >= rect2s[:, 2:])

    # Check for intersection with the "infinitely" extended cross of rect1
    in_cross = ((bl_to_bl[:, 0] <= 0) & (bl_to_tr[:, 0] >= 0)) | ((tr_to_tr[:, 0] >= 0) & (tr_to_bl[:, 0] <= 0)), ((bl_to_bl[:, 1] <= 0) & (bl_to_tr[:, 1] >= 0)) | ((tr_to_tr[:, 1] >= 0) & (tr_to_bl[:, 1] <= 0))

    # Check for equality - if equal, return the original rectangles
    zero = torch.tensor(0, dtype=rect1.dtype, device=rect1.device)
    is_equal = (bl_to_bl.isclose(zero).all(dim=1)) & (tr_to_tr.isclose(zero).all(dim=1))
    print(is_equal)

    # Check for no intersection - if no intersection, return the intersection rectangle [0, 0, 0, 0]
    is_intersecting = inside_tl | inside_br | inside_bl | inside_tr | (enclosure[:, 0] & in_cross[1]) | (enclosure[:, 1] & in_cross[0]) | (enclosure[:, 0] & enclosure[:, 1]) | is_equal

    if not area_only:
        intersections = is_intersecting.unsqueeze(1) * torch.cat((torch.max(rect1[:, :2], rect2s[:, :2]), torch.min(rect1[:, 2:], rect2s[:, 2:])), dim=1)
        intersections[is_equal] = rect1
    else:
        intersections = torch.zeros(rect2s.shape[0], dtype=rect1.dtype, device=rect1.device)
        intersections[is_intersecting] = (torch.min(rect1[:, 2:], rect2s[is_intersecting, 2:]) - torch.max(rect1[:, :2], rect2s[is_intersecting, :2])).abs().prod(dim=1)

    if debug:
        # Used for debugging
        return intersections, torch.stack((inside_tl, inside_tr, inside_br, inside_bl), dim=1), enclosure, in_cross
    else:
        return intersections

In [ ]:
# Example usage
r2 = torch.tensor([2, 2, 5, 5], dtype=dtype, device=device)  # Rectangle 2 coordinates

xmins = [1, 2, 3]
ymins = [1, 2, 3]
xmaxs = [4, 5, 6]
ymaxs = [4, 5, 6]

n_rects = len(xmins) * len(ymins) * len(xmaxs) * len(ymaxs)
ncols = min(6, int(n_rects ** (1/2)))
nrows = math.ceil(n_rects / ncols)

fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))

rectangles = []

for xmin, ymin, xmax, ymax in itertools.product(xmins, ymins, xmaxs, ymaxs):
    r11 = torch.tensor([xmin, ymin, xmax, ymax], dtype=dtype, device=device)
    rectangles.append(r11.clone())

rectangles = torch.stack(rectangles)
rectangles = fix2btlr(rectangles)

intersections, corners, encs, crs = intersect_test(r2.unsqueeze(0), rectangles, debug=True)

# Move tensors to cpu and convert to float
rectangles = rectangles.detach().float().cpu().numpy()
intersections = intersections.detach().float().cpu().numpy()
corners = corners.detach().cpu()
encs = encs.detach().cpu()
crs = torch.stack(crs, dim=1).detach().cpu()
r2 = r2.detach().float().cpu().numpy()

for i, (ax, r1, inter) in enumerate(zip(axs.flatten(), rectangles, intersections)):
    # IOU
    a1 = (r1[2] - r1[0]) * (r1[3] - r1[1])
    a2 = (r2[2] - r2[0]) * (r2[3] - r2[1])
    ai = (inter[2] - inter[0]) * (inter[3] - inter[1])
    iou = ai / (a1 + a2 - ai)

    # Plot rect1
    p1 = mpl.patches.Rectangle((r1[0], r1[1]), r1[2] - r1[0], r1[3] - r1[1], linewidth=3, edgecolor="r", facecolor="none", alpha=1)
    # Plot rect2
    p2 = mpl.patches.Rectangle((r2[0], r2[1]), r2[2] - r2[0], r2[3] - r2[1], linewidth=3, edgecolor="b", facecolor="none", alpha=1)

    # Plot the intersection
    if inter.sum() > 0:
        p3 = mpl.patches.Rectangle((inter[0], inter[1]), inter[2] - inter[0], inter[3] - inter[1], linewidth=0, edgecolor="none", facecolor="g", alpha=0.5)
        ax.add_patch(p3)
        icenter = (inter[2:] + inter[:2]) / 2
        ax.text(icenter[0], icenter[1], f"{iou:.2f}", color="g", fontsize=12, ha="center", va="center")
    ax.add_patch(p1)
    ax.add_patch(p2)

    ax.set_xlim([0, 8])
    ax.set_ylim([0, 8])
    # ax.set_xticks([])
    # ax.set_yticks([])
    ax.set_title("i: " + str(i) + " | co: " + ",".join(["T" if c.item() else "F" for c in corners[i]]) + " | en: " +  ",".join(["T" if e.item() else "F" for e in encs[i]]) + " | cr: " +  ",".join(["T" if c.item() else "F" for c in crs[i]]), fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Test the non-max suppression function
def test_nms(n = 10, thresh=0.5):
    # Generate random boxes
    bxy = torch.rand(n, 2)
    bwh = torch.rand(n, 2) / 4
    boxes = torch.cat((bxy - bwh / 2, bxy + bwh / 2), dim=1)
    # Generate random scores
    scores = torch.rand(n)
    # Run the non-max suppression
    picked_boxes, picked_scores = non_max_suppression(boxes, scores, thresh)
    # Plot the boxes
    fig, ax = plt.subplots(1, figsize=(10, 10))
    for box, conf in zip(boxes, scores):
        rect = mpl.patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=1, edgecolor="none", facecolor="b", alpha=conf.item() / 3)
        ax.add_patch(rect)
    for box, score in zip(picked_boxes, picked_scores):
        rect = mpl.patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=3, edgecolor="r", facecolor="none", alpha=score.item() / 4)
        ax.add_patch(rect)
    plt.show()

    return boxes, scores

tb, ts = test_nms(1000, 0.05)

In [ ]:
tbi = intersect_rect_vectorized(tb[0].unsqueeze(0), tb[1:])

fig, ax = plt.subplots(1, figsize=(10, 10))

for i, (box, conf) in enumerate(zip(tb, ts)):
    if i == 0:
        continue
    rect = mpl.patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=1, edgecolor="none", facecolor="b", alpha=.1)
    ax.add_patch(rect)

tb1 = tb[0]
rect = mpl.patches.Rectangle((tb1[0], tb1[1]), tb1[2] - tb1[0], tb1[3] - tb1[1], linewidth=1, edgecolor="none", facecolor="g", alpha=.5)
ax.add_patch(rect)

for box in tbi:
    rect = mpl.patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], linewidth=1, edgecolor="r", facecolor="none", alpha=1)
    ax.add_patch(rect)

plt.show()

In [ ]:
tb[0]